# Tutorial: DataFrame Records With Custom Metric

Audience:
- Python users with tabular or record-shaped data.

Prerequisites:
- Dictionaries and callable functions.

Learning goals:
- Load DataFrame-like records with stable IDs.
- Define a domain metric over row dictionaries.
- Query records by ID and position.

## Outline

1. Import the public facade.
2. Define a tiny DataFrame-like object.
3. Build a custom metric over rows.
4. Query distances and stable IDs.
5. Try a small exercise.

In [ ]:
from metric import Space
from metric import metrics


## Step 1 - Define records

The facade only needs an object with `to_dict("records")`, so this example avoids a pandas dependency.

In [ ]:
class DataFrameLike:
    def __init__(self, rows):
        self._rows = list(rows)

    def to_dict(self, orient):
        if orient != "records":
            raise ValueError(orient)
        return list(self._rows)

rows = [
    {"sample_id": "pump-a", "temperature": 62.0, "state": "idle"},
    {"sample_id": "pump-b", "temperature": 65.0, "state": "idle"},
    {"sample_id": "valve-c", "temperature": 82.0, "state": "alarm"},
]


## Step 2 - Define a row metric

The metric can combine numeric and categorical fields in domain units.

In [ ]:
def equipment_distance(lhs, rhs):
    state_penalty = 0.0 if lhs["state"] == rhs["state"] else 10.0
    return abs(lhs["temperature"] - rhs["temperature"]) + state_penalty

space = Space.from_dataframe(
    DataFrameLike(rows),
    metric=equipment_distance,
    id_column="sample_id",
    name="equipment",
)

assert space.ids == ["pump-a", "pump-b", "valve-c"]
assert space.record("pump-a") == {"temperature": 62.0, "state": "idle"}
space.name


## Step 3 - Query distances by stable ID

Stable IDs make examples readable and keep record order separate from identity.

In [ ]:
distances = space.pairwise(ids=["pump-a", "valve-c"])

assert distances == [[0.0, 30.0], [30.0, 0.0]]
space.neighbors(space.record("pump-b"), count=1)


## Exercise

Add another idle pump and check that it is closer to `pump-b` than `valve-c` is.

Common pitfall: leaving the ID column in the row metric. `from_dataframe(..., id_column=...)` removes it from records before the metric sees them.

In [ ]:
extended_rows = rows + [{"sample_id": "pump-d", "temperature": 66.0, "state": "idle"}]
extended = Space.from_dataframe(DataFrameLike(extended_rows), metric=equipment_distance, id_column="sample_id")
query_record = {"temperature": 66.5, "state": "idle"}
nearest = extended.neighbors(query_record, count=1)

assert extended.ids[nearest[0][0]] == "pump-d"
nearest
